# **Tugas 2 - Pemrosesan Bahasa Alami**

In [ ]:
"""
/*
 * 245150207111084 Achmad Yusuf Hamdani Firmansyah : Anggota 1 - Corpus Lead & Data Annotation
 * 235150200111040 Putra Cakrawala Aulia Syafiq: Anggota 2 - Core Algorithm (Naïve Bayes Engine)
 * 245150200111056 Zaidan Alvi Zulfan Pramudya: Anggota 3 - Testing Suite & Accuracy Evaluator
 * 245150200111062 Muhammad Azza Arsyada Roziqi: Anggota 4 - Mathematical Verification
 * 245150207111054 Raditya Ramadhan Eka Widhianto: Anggota 5 - Shallow Parsing & NP-Chunking Specialist
 */
"""

## **Import Library**

In [1]:
import os
import re
from collections import defaultdict, Counter
import nltk
from nltk.chunk import RegexpParser
from nltk.chunk import tree2conlltags
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## **Algoritma Naïve Bayes**

In [2]:
class BayesUnigramTagger:
    def __init__(self):
        self.tag_counts = Counter()
        self.word_tag_counts = defaultdict(Counter)
        self.word_counts = Counter()
        self.total_tokens = 0
        self.vocab = set()
        self.all_tags = set()
        self.default_tag = "NN"

    def train_from_file(self, file_path: str):
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File korpus tidak ditemukan di: {file_path}")

        with open(file_path, "r", encoding="utf-8") as f:
            corpus_text = f.read()

        raw_tokens = corpus_text.split()
        parsed_tokens = []

        for item in raw_tokens:
            parts = item.rsplit("/", 1)
            if len(parts) == 2:
                word, tag = parts[0], parts[1]
                parsed_tokens.append((word, tag))

        self.train_from_tokens(parsed_tokens)

    def train_from_tokens(self, tagged_tokens: list):
        self.total_tokens = len(tagged_tokens)
        self.tag_counts.clear()
        self.word_tag_counts.clear()
        self.word_counts.clear()
        self.vocab.clear()
        self.all_tags.clear()

        for word, tag in tagged_tokens:
            self.tag_counts[tag] += 1
            self.word_tag_counts[word][tag] += 1
            self.word_counts[word] += 1
            self.vocab.add(word)
            self.all_tags.add(tag)

        if self.tag_counts:
            self.default_tag = self.tag_counts.most_common(1)[0][0]

    def get_tag_prior(self, tag: str) -> float:
        if self.total_tokens == 0:
            return 0.0
        return self.tag_counts[tag] / self.total_tokens

    def get_lexical_likelihood(self, word: str, tag: str) -> float:
        count_t = self.tag_counts[tag]
        if count_t == 0:
            return 0.0
        return self.word_tag_counts[word][tag] / count_t

    def get_joint_probability(self, word: str, tag: str) -> float:
        return self.get_lexical_likelihood(word, tag) * self.get_tag_prior(tag)

    def get_word_marginal(self, word: str) -> float:
        if self.total_tokens == 0:
            return 0.0
        return self.word_counts[word] / self.total_tokens

    def get_posterior(self, word: str, tag: str) -> float:
        p_w = self.get_word_marginal(word)
        if p_w == 0.0:
            return 0.0
        return self.get_joint_probability(word, tag) / p_w

    def predict_tag(self, word: str):
        if word in self.word_counts:
            best_tag = None
            best_score = -1.0
            for tag in self.all_tags:
                score = self.get_joint_probability(word, tag)
                if score > best_score:
                    best_score = score
                    best_tag = tag
            return best_tag if best_tag is not None else self.default_tag

        w_lower = word.lower()
        lower_tags = Counter()
        for corpus_w, t_counts in self.word_tag_counts.items():
            if corpus_w.lower() == w_lower:
                for t, cnt in t_counts.items():
                    lower_tags[t] += cnt

        if lower_tags:
            best_tag = None
            best_score = -1.0
            for tag, cnt in lower_tags.items():
                likelihood = cnt / self.tag_counts[tag]
                prior = self.get_tag_prior(tag)
                score = likelihood * prior
                if score > best_score:
                    best_score = score
                    best_tag = tag
            return best_tag if best_tag is not None else self.default_tag

        return self.default_tag

    def tokenize(self, text: str) -> list:
        cleaned_text = re.sub(r"\bRp(\d+)", r"Rp \1", text)
        tokens = re.findall(r"\w+(?:-\w+)*|[^\w\s]", cleaned_text)
        return tokens

    def tag_sentence(self, sentence_input) -> list:
        if isinstance(sentence_input, str):
            tokens = self.tokenize(sentence_input)
        else:
            tokens = sentence_input
        return [(word, self.predict_tag(word)) for word in tokens]

    @staticmethod
    def load_raw_test_data(file_path: str) -> list:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File data uji mentah tidak ditemukan di: {file_path}")
        raw_sentences = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                stripped = line.strip()
                if stripped:
                    raw_sentences.append(stripped)
        return raw_sentences

    @staticmethod
    def load_ground_truth_test_data(file_path: str) -> list:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File ground truth tidak ditemukan di: {file_path}")
        ground_truth_sentences = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                stripped = line.strip()
                if not stripped:
                    continue
                tokens_tags = []
                for token_item in stripped.split():
                    parts = token_item.rsplit("/", 1)
                    if len(parts) == 2:
                        tokens_tags.append((parts[0], parts[1]))
                if tokens_tags:
                    ground_truth_sentences.append(tokens_tags)
        return ground_truth_sentences

    def ingest_and_process_test_data(self, raw_path: str, gt_path: str = None) -> list:
        raw_sentences = self.load_raw_test_data(raw_path)
        gt_sentences = self.load_ground_truth_test_data(gt_path) if gt_path else None
        processed_test_data = []
        for idx, raw_sentence in enumerate(raw_sentences):
            tokens = self.tokenize(raw_sentence)
            if gt_sentences and idx < len(gt_sentences):
                gt_sentence = gt_sentences[idx]
                sentence_data = []
                for t_idx, word in enumerate(tokens):
                    gt_tag = gt_sentence[t_idx][1] if t_idx < len(gt_sentence) else self.default_tag
                    sentence_data.append((word, gt_tag))
                processed_test_data.append(sentence_data)
            else:
                processed_test_data.append([(word, None) for word in tokens])
        return processed_test_data

    def evaluate(self, test_data: list):
        total_tokens = 0
        correct_tokens = 0
        results = []
        for idx, sentence in enumerate(test_data, 1):
            sent_correct = 0
            sent_details = []
            for word, true_tag in sentence:
                pred_tag = self.predict_tag(word)
                is_correct = (pred_tag == true_tag) if true_tag is not None else None
                total_tokens += 1
                if is_correct:
                    correct_tokens += 1
                    sent_correct += 1
                sent_details.append({
                    "word": word,
                    "ground_truth": true_tag,
                    "predicted": pred_tag,
                    "correct": is_correct
                })
            sent_acc = (sent_correct / len(sentence)) * 100.0 if sentence else 0.0
            results.append({
                "sentence_idx": idx,
                "length": len(sentence),
                "correct": sent_correct,
                "accuracy": sent_acc,
                "details": sent_details
            })
        overall_acc = (correct_tokens / total_tokens) * 100.0 if total_tokens > 0 else 0.0
        return {
            "total_tokens": total_tokens,
            "correct_tokens": correct_tokens,
            "overall_accuracy": overall_acc,
            "sentences": results
        }

## **Pelatihan & Evaluasi Akurasi**

In [3]:
train_corpus_path = "1_tagged_corpus.txt"
raw_test_path = "test_cases.txt"
gt_test_path = "test_cases_with_ground_truth.txt"

print("=" * 80)
print("TAHAP 1 & 2: TRAINING MODEL & EVALUASI AKURASI")
print("=" * 80)

try:
    # Menginisialisasi dan melatih model
    tagger = BayesUnigramTagger()
    tagger.train_from_file(train_corpus_path)

    print(f"Model berhasil dilatih dari: {train_corpus_path}")
    print(f"Total token (N)            : {tagger.total_tokens}")
    print(f"Jumlah kosakata unik (|V|) : {len(tagger.vocab)}")
    print(f"Jumlah kelas tag unik      : {len(tagger.all_tags)}")

    # Evaluasi Prediksi
    print("\n--- Hasil Evaluasi Prediksi pada Data Uji ---")
    raw_sentences = tagger.load_raw_test_data(raw_test_path)
    test_data = tagger.ingest_and_process_test_data(raw_test_path, gt_test_path)
    eval_result = tagger.evaluate(test_data)

    for res in eval_result["sentences"]:
        print(f"\n>>> Kalimat {res['sentence_idx']} | Akurasi: {res['accuracy']:.2f}%")
        print(f"{'Kata':<15} | {'Ground Truth':<12} | {'Hasil Prediksi':<15} | {'Status':<6}")
        print("-" * 55)
        for item in res["details"]:
            status = "BENAR" if item["correct"] else "SALAH"
            print(f"{item['word']:<15} | {item['ground_truth']:<12} | {item['predicted']:<15} | {status:<6}")

    print("\n--- RINGKASAN AKURASI KESELURUHAN ---")
    print(f"Total token uji      : {eval_result['total_tokens']}")
    print(f"Total prediksi benar : {eval_result['correct_tokens']}")
    print(f"Akurasi Keseluruhan  : {eval_result['overall_accuracy']:.2f}%")

except FileNotFoundError as e:
    print(f"[ERROR] {e}")
    print("Pastikan file 1_tagged_corpus.txt, test_cases.txt, dan test_cases_with_ground_truth.txt ada di folder yang sama dengan file .ipynb ini.")

TAHAP 1 & 2: TRAINING MODEL & EVALUASI AKURASI
Model berhasil dilatih dari: 1_tagged_corpus.txt
Total token (N)            : 1547
Jumlah kosakata unik (|V|) : 703
Jumlah kelas tag unik      : 19

--- Hasil Evaluasi Prediksi pada Data Uji ---

>>> Kalimat 1 | Akurasi: 100.00%
Kata            | Ground Truth | Hasil Prediksi  | Status
-------------------------------------------------------
Presiden        | NNP          | NNP             | BENAR 
Prabowo         | NNP          | NNP             | BENAR 
menyampaikan    | VB           | VB              | BENAR 
kebijakan       | NN           | NN              | BENAR 
baru            | JJ           | JJ              | BENAR 
.               | Z            | Z               | BENAR 

>>> Kalimat 2 | Akurasi: 87.50%
Kata            | Ground Truth | Hasil Prediksi  | Status
-------------------------------------------------------
Pemerintah      | NNP          | NNP             | BENAR 
akan            | MD           | MD              | BENAR 

## **NP-Chunking Specialist**

In [4]:
print("=" * 80)
print("TAHAP 3: SHALLOW PARSING & NP-CHUNKING")
print("=" * 80)

try:
    with open(train_corpus_path, "r", encoding="utf-8") as f:
        teks_korpus = f.readlines()

    kumpulan_kalimat = []
    baris_valid = [b.strip() for b in teks_korpus if b.strip()]
    lima_kalimat_pertama = baris_valid[:5]

    for baris in lima_kalimat_pertama:
        kalimat_tuple = []
        for token in baris.split():
            if '/' in token:
                kata, tag = token.rsplit('/', 1)
                kalimat_tuple.append((kata, tag))
        kumpulan_kalimat.append(kalimat_tuple)

    np_grammar = """
        NP: {<CD|OD>?<NNP|NN|NND>+<JJ>*<PR>?}
    """
    chunk_parser = RegexpParser(np_grammar)

    file_tree = open("2_hasil_np_tree.txt", "w", encoding="utf-8")
    file_iob = open("2_hasil_np_iob.txt", "w", encoding="utf-8")

    for idx, kalimat in enumerate(kumpulan_kalimat, 1):
        tree = chunk_parser.parse(kalimat)

        hasil_tree_str = str(tree)
        file_tree.write(f"Kalimat ke-{idx}:\n{hasil_tree_str}\n\n")
        print(f"\n--- Kalimat ke-{idx} (Parse Tree) ---")
        print(hasil_tree_str)

        iob_tags = tree2conlltags(tree)
        file_iob.write(f"Kalimat ke-{idx}:\n")
        for token in iob_tags:
            file_iob.write(f"{token[0]}\t{token[1]}\t{token[2]}\n")
        file_iob.write("\n")

    file_tree.close()
    file_iob.close()

    print("\n" + "=" * 50)
    print("PROSES SELESAI!")
    print("File '2_hasil_np_tree.txt' & '2_hasil_np_iob.txt' sukses dibuat di folder ini.")
    print("=" * 50)

except FileNotFoundError:
    print(f"\n[ERROR] Gagal membaca korpus di path: {train_corpus_path}. Pastikan file ada di folder yang sama.")

TAHAP 3: SHALLOW PARSING & NP-CHUNKING

--- Kalimat ke-1 (Parse Tree) ---
(S
  Pada/IN
  (NP hari/NND Kamis/NNP)
  ,/Z
  (NP 1/CD Oktober/NNP)
  2026/CD
  ,/Z
  (NP Presiden/NNP Prabowo/NNP Subianto/NNP)
  secara/RB
  resmi/JJ
  melakukan/VB
  (NP perombakan/NN)
  atau/CC
  (NP perombakan/NN kabinet/NN)
  yang/SC
  (NP ketujuh/OD kalinya/NND)
  di/IN
  (NP Istana/NNP Merdeka/NNP)
  ./Z)

--- Kalimat ke-2 (Parse Tree) ---
(S
  Dalam/IN
  (NP
    perombakan/NN
    jajaran/NN
    Kabinet/NNP
    Merah/JJ
    Putih/JJ
    tersebut/PR)
  ,/Z
  (NP sejumlah/CD posisi/NN strategis/JJ)
  mengalami/VB
  (NP pergantian/NN)
  ,/Z
  (NP mulai/NN)
  dari/IN
  (NP tingkatan/NN menteri/NN)
  hingga/IN
  (NP wakil/NN menteri/NN)
  ./Z)

--- Kalimat ke-3 (Parse Tree) ---
(S
  (NP Selain/NN itu/PR)
  ,/Z
  dilantik/VB
  pula/RB
  (NP beberapa/CD pejabat/NN baru/JJ)
  yang/SC
  diharapkan/VB
  mampu/MD
  menambah/VB
  (NP dimensi/NN kebijakan/NN)
  di/IN
  (NP tingkat/NN pemerintahan/NN pusat/NN)
  ./Z)
